In [1]:
from datetime import datetime

from numpy.typing import NDArray
import numpy as np

import clustering
from common import grid_search_no_clusters, cross_validate

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x7f9eb25c9640>>
Traceback (most recent call last):
  File "/home/fzamora/miniconda3/envs/ex/lib/python3.12/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(

KeyboardInterrupt: 


In [ ]:
def get_clusters(adj_matrix: NDArray[np.float64 | np.int32], hyperparameters: dict):
    graph = clustering._adjacency_matrix_to_nxgraph(
        adj_matrix, use_disconnected_edges=False
    )
    clusters = clustering.wsbm_clustering(graph, **hyperparameters)

    return np.array(clustering._convert_graph_cluster_list_set_to_list(graph, clusters))

In [ ]:
def generate_hyperparameters_for_wsbm():
    combinations = []

    for distribution in [
        "discrete-geometric",
        "discrete-poisson",
        "discrete-binomial",
    ]:

        combinations.append(
            {
                "distribution": distribution,
            }
        )

    return combinations

In [ ]:
method = "wsbm"
dataset = "dwug_es"
model = "deepmistake"
path_to_gold_data = "./gold-data-es.csv" 
path_to_data= f"./deepmistake_model_es.csv"

In [ ]:
metadata = {
    "method": method,
    "dataset": dataset,
    "path_to_data": path_to_data,
    "path_to_gold_data": path_to_gold_data,
    "fill_diagonal": True,
    "normalize": True,
    "model": model,
    "use_threshold": True,
    "skip_distribution_filter": True,
}

In [ ]:
start_time = datetime.now()

# grid_search_no_clusters(
#     get_clusters,
#     generate_hyperparameters_for_wsbm(),
#     metadata=metadata,
# )

print(f"Elapsed time {datetime.now() - start_time}")

## Cross-validation experiements

In [ ]:
gold_dir = "./dwug_es_cleaned/clusters"

In [ ]:
start_time = datetime.now()

cv_summary = cross_validate(
    get_clusters,
    generate_hyperparameters_for_wsbm(),
    metadata=metadata,
    gold_dir=gold_dir,
    k=5,
)

print(f"  Elapsed time: {datetime.now() - start_time}")

In [ ]:
print(f"\nProtocol 1 (ARI driven): ")
print(f"  avg test ARI: {cv_summary['protocol_ari']['avg_test_ari']:.4f}")
print(f"  avg test LSCD: {cv_summary['protocol_ari']['avg_test_lscd']:.4f}")
print(f"\nProtocol 2 (LSCD Driven):")
print(f"  avg test LSCD: {cv_summary['protocol_lscd']['avg_test_lscd']:.4f}")
print(f"  avg test ARI: {cv_summary['protocol_lscd']['avg_test_ari']:.4f}")